In [1]:
import pandas as pd
import sqlite3

# 1. Kaggle'dan indirdiğin CSV dosyasını oku
# Veri seti noktalı virgül (;) ile ayrılmış durumda.
df = pd.read_csv('cardio_train.csv', sep=';')

# 2. SQLite veritabanına bağlan
# Bu kod, aynı klasörde 'kalp_hastaligi.db' adında bir veritabanı dosyası oluşturur.
conn = sqlite3.connect('kalp_hastaligi.db')

# 3. Ham veriyi 'raw_data' (ham_veri) adında bir tabloya kaydet
df.to_sql('raw_data', conn, if_exists='replace', index=False)

print("✅ Ham veri başarıyla SQLite veritabanına ('raw_data' tablosuna) kaydedildi!")

# 4. İşlenmiş veriler ve model sonuçları için boş tabloları oluştur
cursor = conn.cursor()

# İşlenmiş veriler için tablo (Bulanıklaştırma işlemi sonrası veriler buraya gelecek)
cursor.execute('''
    CREATE TABLE IF NOT EXISTS processed_data (
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        age INTEGER,
        gender INTEGER,
        other_factors REAL,
        disease_risk REAL
    )
''')

# Model sonuçları için tablo 
cursor.execute('''
    CREATE TABLE IF NOT EXISTS model_results (
        model_name TEXT,
        dataset_type TEXT,
        accuracy REAL,
        computation_time REAL
    )
''')

print("✅ 'processed_data' ve 'model_results' tabloları altyapı olarak oluşturuldu!")

# 5. Bağlantıyı kapat
conn.close()

✅ Ham veri başarıyla SQLite veritabanına ('raw_data' tablosuna) kaydedildi!
✅ 'processed_data' ve 'model_results' tabloları altyapı olarak oluşturuldu!


In [2]:
import pandas as pd
import sqlite3

# 1. Veritabanına tekrar bağlan ve ham veriyi (raw_data) çek
conn = sqlite3.connect('kalp_hastaligi.db')
df = pd.read_sql('SELECT * FROM raw_data', conn)

# Yaş verisi gün cinsinden, bunu makaledeki gibi yıla çeviriyoruz
df['age_years'] = (df['age'] / 365.25).astype(int)

# 2. BULANIKLAŞTIRMA (Fuzzification) İŞLEMİ
# Fiziksel aktivitenin tersini alıyoruz (1 ise 0, 0 ise 1 yapıyoruz)
df['active_neg'] = df['active'].apply(lambda val: 0 if val == 1 else 1)

# Makaledeki Tablo 8 ve Denklem 1 mantığını uygulayan fonksiyon
def calculate_other_factors(row):
    x = row['smoke']
    y = row['alco']
    z_neg = row['active_neg']
    
    # Eğer tüm risk faktörleri yoksa (0, 0, 0)
    if x == 0 and y == 0 and z_neg == 0:
        return 0.0
    # Eğer tüm risk faktörleri varsa (1, 1, 1)
    elif x == 1 and y == 1 and z_neg == 1:
        return 1.0
    # Eğer aralarında bir uyumsuzluk/belirsizlik varsa
    else:
        return 0.5

# Fonksiyonu veri setine uygulayarak 'other_factors' adında tek bir sütun oluştur
df['other_factors'] = df.apply(calculate_other_factors, axis=1)

# 3. İŞLENMİŞ VERİYİ VERİTABANINA KAYDETME
# Hocanın istediği processed_data tablosu için sadece gerekli sütunları alıyoruz
processed_df = pd.DataFrame({
    'age': df['age_years'],
    'gender': df['gender'],
    'other_factors': df['other_factors'],
    'disease_risk': df['cardio'] # Orijinal hastalık durumu
})

# Veritabanına yazdırıyoruz
processed_df.to_sql('processed_data', conn, if_exists='replace', index=False)

print("✅ Bulanıklaştırma (Fuzzification) başarıyla tamamlandı!")
print("-" * 40)
print("İşlenmiş Verinin İlk 5 Satırı (other_factors sütununa dikkat!):")
print(processed_df.head())

conn.close()

✅ Bulanıklaştırma (Fuzzification) başarıyla tamamlandı!
----------------------------------------
İşlenmiş Verinin İlk 5 Satırı (other_factors sütununa dikkat!):
   age  gender  other_factors  disease_risk
0   50       2            0.0             0
1   55       1            0.0             1
2   51       1            0.5             1
3   48       2            0.0             1
4   47       1            0.5             0


In [3]:
import pandas as pd
import sqlite3
import time
import warnings
warnings.filterwarnings('ignore') # Gereksiz uyarıları gizle

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score
from sklearn.naive_bayes import GaussianNB
from sklearn.svm import LinearSVC
from sklearn.ensemble import AdaBoostClassifier, RandomForestClassifier, GradientBoostingClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier

# 1. Veritabanına bağlan ve verileri çek
conn = sqlite3.connect('kalp_hastaligi.db')

# Orijinal Veri (Adil bir karşılaştırma için sadece yaş, cinsiyet ve alışkanlıkları alıyoruz)
raw_df = pd.read_sql('SELECT age, gender, smoke, alco, active, cardio FROM raw_data', conn)
raw_df['age'] = (raw_df['age'] / 365.25).astype(int) 
X_orig = raw_df[['age', 'gender', 'smoke', 'alco', 'active']]
y_orig = raw_df['cardio']

# Modifiye (Bulanık) Veri (Senin oluşturduğun other_factors sütununu içeriyor)
processed_df = pd.read_sql('SELECT age, gender, other_factors, disease_risk FROM processed_data', conn)
X_mod = processed_df[['age', 'gender', 'other_factors']]
y_mod = processed_df['disease_risk']

# Verileri %80 Eğitim, %20 Test olarak makaledeki gibi ayırıyoruz
X_orig_train, X_orig_test, y_orig_train, y_orig_test = train_test_split(X_orig, y_orig, test_size=0.2, random_state=42)
X_mod_train, X_mod_test, y_mod_train, y_mod_test = train_test_split(X_mod, y_mod, test_size=0.2, random_state=42)

# 2. Makaledeki 7 Algoritmayı Tanımla
# Not: Orijinal makalede SVM çok uzun (424 saniye) sürdüğü için, bilgisayarını dondurmamak adına daha hızlı olan LinearSVC versiyonunu kullanıyoruz.
models = {
    "GNB": GaussianNB(),
    "DT": DecisionTreeClassifier(random_state=42),
    "KNN": KNeighborsClassifier(n_neighbors=15),
    "RF": RandomForestClassifier(random_state=42, n_estimators=25),
    "Adaboost": AdaBoostClassifier(random_state=42),
    "GB": GradientBoostingClassifier(random_state=42),
    "SVM": LinearSVC(random_state=42) 
}

results = []

print("🚀 Model eğitimi başlıyor! (Tüm modellerin eğitilmesi 1-2 dakika sürebilir, lütfen arkana yaslan ve bekle...)")

for name, model in models.items():
    print(f"⚙️ Eğitiliyor: {name}...")
    
    # Orijinal Veri Seti Eğitimi ve Ölçümleri
    start_time = time.time()
    model.fit(X_orig_train, y_orig_train)
    orig_preds = model.predict(X_orig_test)
    orig_time = time.time() - start_time
    orig_acc = accuracy_score(y_orig_test, orig_preds)
    orig_prec = precision_score(y_orig_test, orig_preds, zero_division=0)
    
    results.append((name, 'Original', orig_acc, orig_prec, orig_time))
    
    # Modifiye Edilmiş (Bulanık) Veri Seti Eğitimi ve Ölçümleri
    start_time = time.time()
    model.fit(X_mod_train, y_mod_train)
    mod_preds = model.predict(X_mod_test)
    mod_time = time.time() - start_time
    mod_acc = accuracy_score(y_mod_test, mod_preds)
    mod_prec = precision_score(y_mod_test, mod_preds, zero_division=0)
    
    results.append((name, 'Modified', mod_acc, mod_prec, mod_time))

# 3. Sonuçları Veritabanındaki 'model_results' tablosuna kaydet
results_df = pd.DataFrame(results, columns=['model_name', 'dataset_type', 'accuracy', 'precision', 'computation_time'])
results_df.to_sql('model_results', conn, if_exists='replace', index=False)

print("\n✅ Adım 3 Tamamlandı! Tüm modeller eğitildi ve sonuçlar başarıyla veritabanına kaydedildi.")
print("-" * 50)
print("İşte Modellerin Karşılaştırmalı Özeti (İlk 10 Satır):")
print(results_df.sort_values(by=['model_name', 'dataset_type']).head(10))

conn.close()

🚀 Model eğitimi başlıyor! (Tüm modellerin eğitilmesi 1-2 dakika sürebilir, lütfen arkana yaslan ve bekle...)
⚙️ Eğitiliyor: GNB...
⚙️ Eğitiliyor: DT...
⚙️ Eğitiliyor: KNN...
⚙️ Eğitiliyor: RF...
⚙️ Eğitiliyor: Adaboost...
⚙️ Eğitiliyor: GB...
⚙️ Eğitiliyor: SVM...

✅ Adım 3 Tamamlandı! Tüm modeller eğitildi ve sonuçlar başarıyla veritabanına kaydedildi.
--------------------------------------------------
İşte Modellerin Karşılaştırmalı Özeti (İlk 10 Satır):
   model_name dataset_type  accuracy  precision  computation_time
9    Adaboost     Modified  0.598429   0.610423          0.650820
8    Adaboost     Original  0.600714   0.604313          0.750883
3          DT     Modified  0.598214   0.599512          0.033957
2          DT     Original  0.599857   0.599073          0.047605
11         GB     Modified  0.598857   0.597160          2.105269
10         GB     Original  0.600214   0.600255          2.060139
1         GNB     Modified  0.595643   0.583695          0.015835
0         G